# MANGO / FairCare-FL - Colab GPU Runner

Runs the fair-federated-learning research pipeline on a Colab **GPU** (CUDA).

**Before running:** `Runtime > Change runtime type > Hardware accelerator: GPU`, then `Runtime > Run all`.

This notebook: clones `main`, installs deps, checks the GPU, runs a smoke test, the
validation gate (proves results are non-degenerate), a controlled-bias check, and the new
dataset loaders. The full sweep is guarded behind a flag near the bottom. See `HANDOFF.md`
for the full context and roadmap.

## 1. GPU check

In [ ]:
import torch
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('NO GPU -> Runtime > Change runtime type > GPU, then Run all again')

## 2. Clone the repo + install

In [ ]:
import os
os.chdir('/content')
if not os.path.exists('/content/mango'):
    get_ipython().system('git clone -b main https://github.com/muzakkirhussain011/mango.git')
os.chdir('/content/mango')
get_ipython().system('git pull --ff-only')
print('CWD:', os.getcwd())

In [ ]:
# Installs faircare + all deps (incl. cvxpy, required by the FairCare-FL aggregator).
get_ipython().system('pip install -e . -q')
print('install done')

## 3. Import check + run helper

In [ ]:
import sys, subprocess, glob, json, os
import numpy as np
import pandas as pd
from faircare.experiments.run_experiments import _select_device
import faircare.algos.faircare_fl  # forces the cvxpy import path to resolve
print('device ->', _select_device('auto'))

def run_experiment(algorithm, dataset, sensitive_attr='sex', rounds=30, num_clients=10,
                   local_epochs=1, seed=0, device='cuda', extra=None, save_root='results/colab'):
    save_dir = f'{save_root}/{dataset}/{algorithm}/seed{seed}'
    cmd = [sys.executable, '-m', 'faircare.experiments.run_experiments',
           '--dataset', dataset, '--algorithm', algorithm, '--sensitive_attr', sensitive_attr,
           '--rounds', str(rounds), '--num_clients', str(num_clients),
           '--local_epochs', str(local_epochs), '--seed', str(seed),
           '--device', device, '--save_dir', save_dir]
    if extra:
        cmd += extra
    print('[RUN]', algorithm, dataset, 'seed', seed, flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:])
        print(r.stderr[-3000:])
        raise RuntimeError(f'{algorithm}/{dataset} failed')
    files = sorted(glob.glob(f'{save_dir}/**/final_results.json', recursive=True), key=os.path.getmtime)
    with open(files[-1]) as fh:
        res = json.load(fh)
    m = res['final_metrics']
    return {'algorithm': algorithm, 'dataset': dataset, 'seed': seed,
            'accuracy': m['test/accuracy'], 'auroc': m['test/auroc'],
            'worst_group_f1': m['test/worst_group_f1'],
            'eo_gap': m['test/eo_gap'], 'fpr_gap': m['test/fpr_gap'], 'sp_gap': m['test/sp_gap'],
            'time_s': round(res['total_time'], 1)}
print('helper ready')

## 4. Smoke test (fast, proves the pipeline end-to-end)

In [ ]:
r = run_experiment('fedavg', 'adult', 'sex', rounds=5, seed=0)
print('SMOKE OK:', r)

## 5. Validation gate (Adult, all 6 algorithms)
AUROC must be ~0.85-0.90 (NOT ~0.5), and fairness gaps must be non-zero and vary by algorithm.

In [ ]:
algos = ['fedavg', 'fedprox', 'qffl', 'afl', 'fairfed', 'faircare_fl']
rows = [run_experiment(a, 'adult', 'sex', rounds=30, seed=0) for a in algos]
df = pd.DataFrame(rows).set_index('algorithm')[['accuracy','auroc','worst_group_f1','eo_gap','fpr_gap','sp_gap','time_s']]
display(df.round(4))

auroc_ok = df.loc['fedavg','auroc'] > 0.80
gaps = df[['eo_gap','fpr_gap','sp_gap']].to_numpy()
gaps_ok = float(np.nanstd(gaps)) > 1e-4 and float(np.nanmax(gaps)) > 1e-3
print()
print('VALIDATION GATE (single-seed quick check):')
print(('[OK]' if auroc_ok else '[FAIL]'), 'Adult FedAvg AUROC > 0.80  (got %.3f)' % df.loc['fedavg','auroc'])
print(('[OK]' if gaps_ok else '[FAIL]'), 'Fairness gaps non-zero and vary across algorithms')
print('NOTE: for the full gate run >=3 seeds and confirm per-seed accuracy std > 0 (see HANDOFF.md).')

## 6. Controlled-bias check (synth_health)
On data with a KNOWN injected bias, FairCare-FL should reduce the EO gap vs FedAvg.

In [ ]:
sh = pd.DataFrame([run_experiment(a, 'synth_health', rounds=30, seed=0) for a in ['fedavg','faircare_fl']]).set_index('algorithm')
display(sh[['accuracy','auroc','worst_group_f1','eo_gap']].round(4))
better = sh.loc['faircare_fl','eo_gap'] < sh.loc['fedavg','eo_gap']
print(('[OK]' if better else '[CHECK]'), 'FairCare-FL EO gap < FedAvg EO gap on synth_health')

## 7. Validate the NEW real loaders (first run downloads + caches)

In [ ]:
for ds, sattr in [('diabetes130','race'), ('compas','race')]:
    r = run_experiment('fedavg', ds, sattr, rounds=10, seed=0)
    print(ds, '->', {k: r[k] for k in ('accuracy','auroc','worst_group_f1','eo_gap')})

## 8. Full headline sweep (optional - takes a while)
Set `RUN_FULL_SWEEP = True` to run 6 algos x 4 datasets x 5 seeds and write a summary CSV.

In [ ]:
RUN_FULL_SWEEP = False  # <- set True for the headline grid
if RUN_FULL_SWEEP:
    datasets = [('adult','sex'), ('diabetes130','race'), ('compas','race'), ('synth_health','sex')]
    algos = ['fedavg','fedprox','qffl','afl','fairfed','faircare_fl']
    seeds = [0, 1, 2, 3, 4]
    allrows = []
    for ds, sa in datasets:
        for a in algos:
            for s in seeds:
                allrows.append(run_experiment(a, ds, sa, rounds=50, local_epochs=2,
                                              num_clients=20, seed=s, extra=['--dirichlet_alpha','0.3']))
    full = pd.DataFrame(allrows)
    os.makedirs('results/colab', exist_ok=True)
    full.to_csv('results/colab/full_sweep_raw.csv', index=False)
    summary = full.groupby(['dataset','algorithm']).mean(numeric_only=True).round(4)
    display(summary)
    summary.to_csv('results/colab/full_sweep_summary.csv')
    print('Saved results/colab/full_sweep_*.csv')
else:
    print('RUN_FULL_SWEEP is False. Set it to True to run the headline grid.')

## 9. Download results

In [ ]:
import shutil
if os.path.exists('results/colab'):
    shutil.make_archive('mango_results', 'zip', 'results/colab')
    try:
        from google.colab import files
        files.download('mango_results.zip')
    except Exception as e:
        print('Zip written to /content/mango/mango_results.zip (download manually):', e)
else:
    print('No results/colab dir yet - run the cells above first.')